# **Notebook 02: Pré-processamento e Feature Engineering**

Nesta etapa, saímos da análise visual (EDA) e preparamos os dados de fato para o modelo de Machine Learning. O objetivo aqui é limpar o dataset e transformar todas as variáveis em formatos numéricos.

### **1. Carga dos Dados**
Vamos importar o dataset `dataset_eda_concluido.csv` que exportamos no final do Notebook 01, já com as features comportamentais que criamos (horário, tamanho, extensões, etc).

In [1]:
import pandas as pd
from google.colab import drive

# Montando o Google Drive
drive.mount('/content/drive')

# Definindo o caminho do dataset gerado no Notebook 01
path_input = '/content/drive/MyDrive/Colab_Notebooks/ml_bot_detector/dataset_eda_concluido.csv'

# Carregando os dados
df = pd.read_csv(path_input)

print("--- DATASET CARREGADO ---")
print(f"Total de registros: {df.shape[0]}")
print(f"Colunas disponíveis: {df.columns.tolist()}")
df.head(3)

Mounted at /content/drive
--- DATASET CARREGADO ---
Total de registros: 199908
Colunas disponíveis: ['ip', 'data', 'metodo', 'url', 'status', 'tamanho', 'referer', 'user_agent', 'Target', 'tamanho_num', 'hora', 'extensao', 'is_suspicious_path']


,ip,data,metodo,url,status,tamanho,referer,user_agent,Target,tamanho_num,hora,extensao,is_suspicious_path
0,/opt/bitnami/apache/logs/access_log-20250928.g...,21/Sep/2025:01:47:50 +0000,GET,/tag/carreira/page/8/?nowprocket=1&no_optimize...,200,39660,-,-,1,39660,1,sem_extensao,0
1,/opt/bitnami/apache/logs/access_log-20250928.g...,21/Sep/2025:01:47:51 +0000,GET,/wp-content/plugins/cookie-law-info/legacy/pub...,200,953,-,-,1,953,1,css,0
2,/opt/bitnami/apache/logs/access_log-20250928.g...,21/Sep/2025:01:47:51 +0000,GET,/wp-content/themes/Newspaper-child/style.css?v...,200,359,-,-,1,359,1,css,0


### **2. Seleção de Features e Ajuste de Viés Temporal**

Para evitar *overfitting* e garantir que o modelo aprenda padrões de comportamento reais, descartamos colunas de identificação direta (como IP e URL bruta).

Além disso, para evitar um "viés temporal" (fazer o modelo decorar que bots atacam em uma hora específica, como 14h), vamos substituir a feature `hora` pela feature `req_por_minuto`. Isso ensina ao algoritmo a identificar a **intensidade** do tráfego (picos de automação), combinando isso com as outras features fundamentais (tamanho, status, extensões e caminhos sensíveis).

In [2]:
# Criando a feature de Intensidade (Requisições por Minuto)
# Extraímos a string da data até a casa do minuto (Ex: '[22/Mar/2026:14:30')
df['data_minuto'] = df['data'].astype(str).str[:17]

# Contamos quantas requisições ocorreram naquele exato mesmo minuto no dataset
df['req_por_minuto'] = df.groupby('data_minuto')['data_minuto'].transform('count')

# Definindo as colunas comportamentais (Features) e o Alvo (Target)
# Nota: Substituímos 'hora' por 'req_por_minuto', mantendo todas as outras essenciais
cols_modelo = ['status', 'tamanho_num', 'req_por_minuto', 'is_suspicious_path', 'extensao', 'metodo', 'Target']
df_modelo = df[cols_modelo].copy()

# Forçando a tipagem correta e preenchendo possíveis nulos gerados na conversão
df_modelo['status'] = pd.to_numeric(df_modelo['status'], errors='coerce').fillna(0).astype(int)
df_modelo['tamanho_num'] = df_modelo['tamanho_num'].fillna(0)

print("--- VERIFICAÇÃO DE VALORES NULOS ---")
print(df_modelo.isnull().sum())

--- VERIFICAÇÃO DE VALORES NULOS ---
status                0
tamanho_num           0
req_por_minuto        0
is_suspicious_path    0
extensao              1
metodo                0
Target                0
dtype: int64


### **3. Tratamento de Variáveis Categóricas (One-Hot Encoding)**

Modelos baseados em árvores ou regressão não compreendem strings como "GET", "POST", ".js" ou ".css". Eles dependem de matrizes numéricas.

Para resolver isso, aplicamos a técnica de **One-Hot Encoding**. Essa técnica converte categorias em múltiplas colunas binárias (0 ou 1). Por exemplo, a requisição ser `.js` ativa a coluna `extensao_js` com o valor 1, mantendo as outras zeradas. Utilizamos o parâmetro `drop_first=True` para descartar a primeira coluna gerada de cada categoria, evitando redundância e a "armadilha da variável dummy" (multicolinearidade).

In [3]:
# Transformando categorias de texto em colunas binárias numéricas
df_final = pd.get_dummies(df_modelo, columns=['extensao', 'metodo'], drop_first=True)

print("--- SHAPE APÓS O ONE-HOT ENCODING ---")
print(f"O dataset agora possui {df_final.shape[1]} colunas puramente numéricas.")

# Exibindo como ficaram as novas colunas binárias
df_final.head(3)

--- SHAPE APÓS O ONE-HOT ENCODING ---
O dataset agora possui 56 colunas puramente numéricas.


,status,tamanho_num,req_por_minuto,is_suspicious_path,Target,extensao_alfa,extensao_asp,extensao_aspx,extensao_aws,extensao_axd,...,extensao_woff,extensao_xml,extensao_yml,extensao_zip,metodo_GET,metodo_HEAD,metodo_OPTIONS,metodo_POST,metodo_PROPFIND,metodo_SSTP_DUPLEX_POST
0,200,39660,128,0,1,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
1,200,953,128,0,1,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
2,200,359,128,0,1,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False


### **4. Separação de Features (X) e Target (y) e Divisão Treino/Teste**

Agora que o dataset é puramente numérico, vamos separar a nossa variável alvo (`Target`, que diz se é bot ou humano) do restante das características (Features - `X`).

Em seguida, dividimos os dados usando o `train_test_split` (80% para treino e 20% para teste). O uso do parâmetro `stratify=y` é crucial aqui para garantir que a mesma proporção de bots e humanos seja mantida em ambos os conjuntos, evitando que o modelo treine de forma enviesada.

In [4]:
from sklearn.model_selection import train_test_split

# Separando as Features (X) e o Alvo (y)
X = df_final.drop('Target', axis=1)
y = df_final['Target']

# Dividindo em Treino (80%) e Teste (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y # Mantém a proporção de classes
)

print("--- SHAPE DOS CONJUNTOS DE TREINO E TESTE ---")
print(f"Treino (X_train): {X_train.shape[0]} amostras")
print(f"Teste (X_test): {X_test.shape[0]} amostras")
print(f"Total de colunas (features): {X_train.shape[1]}")

--- SHAPE DOS CONJUNTOS DE TREINO E TESTE ---
Treino (X_train): 159926 amostras
Teste (X_test): 39982 amostras
Total de colunas (features): 55


### **5. Exportação dos Dados para Modelagem**

Para mantermos a arquitetura do projeto modularizada, vamos exportar os conjuntos de treino e teste separados.

Isso garante que o próximo notebook (focado exclusivamente no treinamento do algoritmo e avaliação das métricas) possa ser executado de forma independente, sem a necessidade de reprocessar toda a engenharia de features.

In [5]:
# Definindo os caminhos para salvar cada arquivo no Drive
caminho_base = '/content/drive/MyDrive/Colab_Notebooks/ml_bot_detector/'

print("Salvando os conjuntos de dados. Isso pode levar alguns segundos...")

# Exportando X e y de treino
X_train.to_csv(f'{caminho_base}X_train.csv', index=False)
y_train.to_csv(f'{caminho_base}y_train.csv', index=False)

# Exportando X e y de teste
X_test.to_csv(f'{caminho_base}X_test.csv', index=False)
y_test.to_csv(f'{caminho_base}y_test.csv', index=False)

print("✅ Todos os datasets foram salvos com sucesso e estão prontos para o Notebook 03!")

Salvando os conjuntos de dados. Isso pode levar alguns segundos...
✅ Todos os datasets foram salvos com sucesso e estão prontos para o Notebook 03!
